### 앙상블 
- 단일 결정트리(DecisionTree)의 단점을 극복하기 위해 여러 머신러닝 모델을 연결하여 더 강력한 모델을 만드는 과정 
- 주어진 자료로부터 여러개의 예측 모형을 만든 후 예측 모형을 조합하여 하나의 예측 모형을 생성하는 과정 
- 대표적인 기법 : 배깅, 부스팅, 랜덤포레스트


#### 배깅 
- 주어진 자료를 모집단으로 간주하여 주어진 자료에서 여러 개의 부트스트랩 자료를 생성하고 각각의 부트스트랩 자료에 예측 모형을 만든 후 결합하여 최종 모델 완성 
- 장점 
    - 분산을 줄이고 과적합의 문제를 완화
    - 안정적인 성능 보장 
    - 비선형 구조의 데이터와 노이즈 데이터에 대한 문제가 완화
- parameter 
    - estimator
        - 기본값 : None
        - 기본 모델을 설정
    - n_estimator
        - 기본값 : 10
        - 생성시킬 모델의 개수
    - max_samples
        - 기본값 : 1.0
        - 생성이 된 모델들이 사용할 데이터의 개수(인덱스의 수) 비율
    - max_features
        - 기본값 : 1.0
        - 생성이 된 모델들이 사용할 피쳐(컬럼)의 수 비율
    - oob_score
        - 기본값 : False
        - oob(Out-Of-Bag) 데이터로 일반화 성능을 평가할것인가?
    - bootstrap
        - 기본값 : True
        - 샘플링 데이터를 이용 시 중복 데이터를 허용
        - False인 경우에는 각각의 모델들에 max_samples의 비율이 1.0이라면 모두 같은 데이터를 학습으로 사용( 다양성이 부족 )
    - bootstrap_features
        - 기본값 : False
        - 샘플링 데이터 이용시 중복 컬럼을 허용할것인가
    - n_jobs
        - 기본값 : None
        - CPU의 병렬처리 개수( -1을 사용시에는 모든 CPU를 사용 )
- 속성 
    - estimators_
        - 학습된 모델의 리스트
    - estimators_samples_
        - 각 모델이 학습한 샘플의 인덱스(행의 위치)
    - estimators_features_
        - 각 모델이 학습한 컬럼의 인덱스(열의 위치)
    - oob_score_
        - oob 데이터를 기반으로 정확도(분류) / R2(회귀)
    - oob_decision_function_
        - OOB 데이터의 클래스별 예측 확률
- 메서드 
    - fit_predict()
        - 학습 후 예측 수행 (학습 데이터로 예측)

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import f1_score

In [2]:
hotel = pd.read_csv("../data/hotel_bookings.csv")

In [3]:
hotel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     20000 non-null  int64  
 1   deposit_type                    20000 non-null  object 
 2   lead_time                       19995 non-null  float64
 3   stays_in_weekend_nights         20000 non-null  int64  
 4   stays_in_week_nights            20000 non-null  int64  
 5   is_repeated_guest               19642 non-null  float64
 6   previous_cancellations          20000 non-null  int64  
 7   previous_bookings_not_canceled  20000 non-null  int64  
 8   booking_changes                 20000 non-null  int64  
 9   days_in_waiting_list            20000 non-null  int64  
 10  adr                             18937 non-null  float64
dtypes: float64(3), int64(7), object(1)
memory usage: 1.7+ MB


- is_canceled : 예약의 취소 여부 ( 1 = 취소, 0 = 체크인 ) (target 데이터)
- deposit_type : 보증금의 유형 (문자열) --> 더미변수 생성
- lead_time : 예약 시차 ( 예약일 ~ 체크인일 )
- stays_in_weekend_nights : 주말 숙박 일수
- stays_in_week_nights : 평일 숙박 일수
- is_repeated_guest : 재방문 고객 여부 (1 = 재방문, 0 = 신규)
- previous_cancellations : 과거 취소 횟수
- previous_bookings_not_canceled : 과거 정상 투숙 횟수
- booking_changes : 예약 변경 횟수
- days_in_waiting_list : 대기 명단에 있었던 횟수
- adr : 1박당 평균 객실의 요금

In [5]:
hotel['lead_time'].describe()

count    19995.000000
mean        85.978345
std         96.427240
min          0.000000
25%         11.000000
50%         51.000000
75%        132.000000
max        629.000000
Name: lead_time, dtype: float64

In [6]:
hotel['is_repeated_guest'].describe()

count    19642.000000
mean         0.038133
std          0.191521
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: is_repeated_guest, dtype: float64

In [7]:
hotel['is_repeated_guest'].value_counts()

is_repeated_guest
0.0    18893
1.0      749
Name: count, dtype: int64

In [8]:
hotel['adr'].describe()

count    18937.000000
mean       101.410239
std         49.245097
min         -6.380000
25%         68.800000
50%         94.500000
75%        126.000000
max        451.500000
Name: adr, dtype: float64

In [10]:
# adr 데이터에서 음수인 데이터를 확인 
flag = hotel['adr'] < 0
hotel = hotel.loc[~flag, ]

In [11]:
hotel['adr'].describe()

count    18936.000000
mean       101.415931
std         49.240166
min          0.000000
25%         68.822500
50%         94.500000
75%        126.000000
max        451.500000
Name: adr, dtype: float64

In [12]:
hotel.loc[ hotel['adr'] == 0,  ]

,is_canceled,deposit_type,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
42,0,No Deposit,97.0,0,2,0.0,0,0,0,0,0.0
46,0,No Deposit,0.0,0,1,1.0,0,3,0,0,0.0
80,0,No Deposit,8.0,0,0,0.0,0,0,0,0,0.0
108,0,No Deposit,30.0,2,3,0.0,0,0,4,14,0.0
119,0,No Deposit,4.0,0,2,0.0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
18558,1,No Deposit,244.0,2,4,0.0,0,0,0,0,0.0
19141,1,No Deposit,71.0,1,1,0.0,0,0,0,0,0.0
19223,1,No Deposit,1.0,0,1,1.0,4,11,0,0,0.0
19253,1,No Deposit,23.0,1,1,0.0,0,0,1,0,0.0


In [13]:
# lead_time, adr 컬럼의 결측치들은 평균값 | 중간값으로 대체 
hotel['lead_time'] = hotel['lead_time'].fillna( hotel['lead_time'].mean() )
hotel['adr'] = hotel['adr'].fillna(hotel['adr'].mean())

In [14]:
# 결측치 개수 확인 isna() + sum()
hotel.isna().sum()

is_canceled                         0
deposit_type                        0
lead_time                           0
stays_in_weekend_nights             0
stays_in_week_nights                0
is_repeated_guest                 358
previous_cancellations              0
previous_bookings_not_canceled      0
booking_changes                     0
days_in_waiting_list                0
adr                                 0
dtype: int64

In [17]:
hotel['is_repeated_guest'].value_counts().index[0]

np.float64(0.0)

In [18]:
# is_repeated_guest 컬럼의 결측치는 해당 컬럼의 최빈값으로 채워준다. ( 범주형 데이터이기때문에 )
hotel['is_repeated_guest'] = hotel['is_repeated_guest'].fillna(
    hotel['is_repeated_guest'].value_counts().index[0]
)

In [19]:
# deposit_type 컬럼은 문자형 데이터 -> 문자의 유형들을 확인 
hotel['deposit_type'].unique()

array(['No Deposit', 'Refundable', 'Non Refund'], dtype=object)

In [20]:
hotel['deposit_type'].value_counts()

deposit_type
No Deposit    19137
Non Refund      834
Refundable       28
Name: count, dtype: int64

In [21]:
df = pd.get_dummies(hotel, columns=['deposit_type'], drop_first=True)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19999 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     19999 non-null  int64  
 1   lead_time                       19999 non-null  float64
 2   stays_in_weekend_nights         19999 non-null  int64  
 3   stays_in_week_nights            19999 non-null  int64  
 4   is_repeated_guest               19999 non-null  float64
 5   previous_cancellations          19999 non-null  int64  
 6   previous_bookings_not_canceled  19999 non-null  int64  
 7   booking_changes                 19999 non-null  int64  
 8   days_in_waiting_list            19999 non-null  int64  
 9   adr                             19999 non-null  float64
 10  deposit_type_Non Refund         19999 non-null  bool   
 11  deposit_type_Refundable         19999 non-null  bool   
dtypes: bool(2), float64(3), int64(7)
memo

In [23]:
df['is_canceled'].value_counts()

is_canceled
0    17599
1     2400
Name: count, dtype: int64

In [24]:
# 데이터의 불균형을 유지한채 독립 , 종속 변수 
x = df.drop('is_canceled', axis=1)
y = df['is_canceled']

In [25]:
# train, test 데이터셋으로 나눠준다 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [26]:
y_train.value_counts()

is_canceled
0    14079
1     1920
Name: count, dtype: int64

In [43]:
# 배깅 모델 생성 -> DecisionTree를 기본 모델로 사용 --> 데이터의 불균형이 심하다 --> class_weight
base_model = DecisionTreeClassifier(class_weight='balanced', max_depth=3)

model = BaggingClassifier(base_model, n_estimators=500)

In [44]:
model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.","DecisionTreeC..., max_depth=3)"
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",500
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [45]:
pred = model.predict(X_test)

In [46]:
print(
    round( f1_score(y_test, pred), 4 )
)

0.5748


- 데이터의 불균형을 샘플링 기법을 통해서 리샘플을 하고 모델에 학습 시켜서 성능을 확인 
- 랜덤오버샘플링 -> 2:1비율로 리샘플링
- 배깅 모델에서 부트스트랩에서 샘플의 개수를 0.8로 지정하고 모델의 개수는 100
- 의사결정나무에서 class_weight 기본값으로 최대 깊이는 3 사용하여 성능을 평가 
